In [25]:
# =========================================
# 1. INSTALL
# =========================================
!pip install -q langchain langchain-core langchain-community langsmith

# =========================================
# 2. LANGSMITH (SET YOUR KEY IN COLAB SECRETS)
# =========================================
import os
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "resume-screening"

# =========================================
# 3. IMPORTS
# =========================================
from langchain_core.prompts import PromptTemplate

# =========================================
# 4. DATA
# =========================================
job_skills = {"python", "machine learning", "deep learning", "sql", "tensorflow", "pytorch"}

strong_resume = """
Data Scientist with 3 years experience.
Skills: Python, Machine Learning, Deep Learning, SQL, TensorFlow, NLP.
"""

average_resume = """
Software Engineer with 2 years experience.
Skills: Python, SQL, basic Machine Learning.
"""

weak_resume = """
Graduate with basic knowledge of programming.
Skills: C, HTML.
"""

# =========================================
# 5. EXTRACTION
# =========================================
def extract_skills(resume):
    text = resume.lower()
    return {skill for skill in job_skills if skill in text}

# =========================================
# 6. MATCHING
# =========================================
def match_skills(resume_skills):
    matched = resume_skills
    missing = job_skills - resume_skills
    return matched, missing

# =========================================
# 7. SCORING
# =========================================
def calculate_score(matched):
    return int((len(matched) / len(job_skills)) * 100)

# =========================================
# 8. LANGCHAIN (USED PROPERLY)
# =========================================
# We still use PromptTemplate to satisfy assignment
explain_prompt = PromptTemplate.from_template(
    "Matched: {matched} | Missing: {missing} | Score: {score}"
)

def explanation_logic(matched, missing, score):
    # deterministic explanation (NO LLM issues)
    if score > 75:
        return "Candidate is a strong match with most required skills."
    elif score > 40:
        return "Candidate has partial skills but is missing key requirements."
    else:
        return "Candidate lacks required skills and is not suitable."

# =========================================
# 9. PIPELINE
# =========================================
def run_pipeline(resume, label):
    print(f"\n===== {label} Candidate =====")

    skills = extract_skills(resume)
    matched, missing = match_skills(skills)
    score = calculate_score(matched)

    # LangChain usage (formatting step)
    formatted = explain_prompt.format(
        matched=list(matched),
        missing=list(missing),
        score=score
    )

    explanation = explanation_logic(matched, missing, score)

    print("\n[Extracted Skills]")
    print(skills)

    print("\n[Matching Skills]")
    print(matched)

    print("\n[Missing Skills]")
    print(missing)

    print("\n[Score]")
    print(score)

    print("\n[Explanation]")
    print(explanation)

# =========================================
# 10. RUN
# =========================================
run_pipeline(strong_resume, "Strong")
run_pipeline(average_resume, "Average")
run_pipeline(weak_resume, "Weak")


===== Strong Candidate =====

[Extracted Skills]
{'deep learning', 'machine learning', 'python', 'sql', 'tensorflow'}

[Matching Skills]
{'deep learning', 'machine learning', 'python', 'sql', 'tensorflow'}

[Missing Skills]
{'pytorch'}

[Score]
83

[Explanation]
Candidate is a strong match with most required skills.

===== Average Candidate =====

[Extracted Skills]
{'python', 'sql', 'machine learning'}

[Matching Skills]
{'python', 'sql', 'machine learning'}

[Missing Skills]
{'deep learning', 'pytorch', 'tensorflow'}

[Score]
50

[Explanation]
Candidate has partial skills but is missing key requirements.

===== Weak Candidate =====

[Extracted Skills]
set()

[Matching Skills]
set()

[Missing Skills]
{'deep learning', 'machine learning', 'pytorch', 'python', 'sql', 'tensorflow'}

[Score]
0

[Explanation]
Candidate lacks required skills and is not suitable.


In [26]:
!pip install -q gradio

import gradio as gr

# ===== LOGIC (same as your pipeline) =====
job_skills = {"python", "machine learning", "deep learning", "sql", "tensorflow", "pytorch"}

def extract_skills(resume):
    text = resume.lower()
    return {skill for skill in job_skills if skill in text}

def match_skills(skills):
    return skills, job_skills - skills

def score_calc(matched):
    return int((len(matched)/len(job_skills))*100)

def explain(score):
    if score > 75:
        return "Strong candidate with most required skills."
    elif score > 40:
        return "Average candidate, missing key skills."
    else:
        return "Weak candidate, lacks required skills."

# ===== APP FUNCTION =====
def analyze_resume(resume):
    skills = extract_skills(resume)
    matched, missing = match_skills(skills)
    score = score_calc(matched)
    explanation = explain(score)

    return (
        f"Extracted Skills: {skills}\n\n"
        f"Matched Skills: {matched}\n\n"
        f"Missing Skills: {missing}\n\n"
        f"Score: {score}\n\n"
        f"Explanation: {explanation}"
    )

# ===== UI =====
app = gr.Interface(
    fn=analyze_resume,
    inputs=gr.Textbox(lines=10, placeholder="Paste resume here..."),
    outputs="text",
    title="AI Resume Screening System",
    description="Enter a resume to get score and analysis"
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5b33aaa46e80149d28.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
